# Storage of results in the database

The duckDB database will contain the following tables
- nav
- holdings
- targets
- trades
- assets returns (because prices are revised this will make NAV checkable)

In [1]:
import duckdb
import pandas as pd
import tempfile
from datetime import date
import pytest
from pathlib import Path
from vportfolio.params import LIVE_DB

In [2]:
con = duckdb.connect() # no file name: a temporary database in memory

In [3]:
# create the nav table
con.execute("""
    CREATE TABLE IF NOT EXISTS nav (
        date DATE,
        portfolio VARCHAR,
        nav DOUBLE,
        daily_return DOUBLE,
        PRIMARY KEY (date, portfolio)
    )
""")

In [4]:
# create the holdings table
con.execute("""
    CREATE TABLE IF NOT EXISTS holdings (
        date DATE,
        portfolio VARCHAR,
        ticker VARCHAR,
        weight DOUBLE,
        PRIMARY KEY (date, portfolio, ticker)
    )
""")

In [5]:
# create the targets table
con.execute("""
    CREATE TABLE IF NOT EXISTS targets (
        decision_date DATE,
        portfolio VARCHAR,
        ticker VARCHAR,
        weight DOUBLE,
        PRIMARY KEY (decision_date, portfolio, ticker)
    )
""")

In [6]:
# create the trades table
con.execute("""
    CREATE TABLE IF NOT EXISTS trades (
        trade_date DATE,
        portfolio VARCHAR,
        ticker VARCHAR,
        weight_before DOUBLE,
        weight_after DOUBLE,
        cost DOUBLE,
        PRIMARY KEY (trade_date, portfolio, ticker)
    )
""")

In [7]:
# create the asset returns table
con.execute("""
    CREATE TABLE IF NOT EXISTS asset_returns (
        date DATE,
        ticker VARCHAR,
        daily_return DOUBLE,
        PRIMARY KEY (date, ticker)
    )
""")

In [8]:
# database trial
con.execute("SHOW TABLES").df()

,name
0,asset_returns
1,holdings
2,nav
3,targets
4,trades


In [9]:
# now we chain these into a function
def create_tables(con):
    # create the nav table
    con.execute("""
        CREATE TABLE IF NOT EXISTS nav (
            date DATE,
            portfolio VARCHAR,
            nav DOUBLE,
            daily_return DOUBLE,
            PRIMARY KEY (date, portfolio)
        )
    """)
    # create the holdings table
    con.execute("""
        CREATE TABLE IF NOT EXISTS holdings (
            date DATE,
            portfolio VARCHAR,
            ticker VARCHAR,
            weight DOUBLE,
            PRIMARY KEY (date, portfolio, ticker)
        )
    """)
    # create the targets table
    con.execute("""
        CREATE TABLE IF NOT EXISTS targets (
            decision_date DATE,
            portfolio VARCHAR,
            ticker VARCHAR,
            weight DOUBLE,
            PRIMARY KEY (decision_date, portfolio, ticker)
        )
    """)
    # create the trades table
    con.execute("""
        CREATE TABLE IF NOT EXISTS trades (
            trade_date DATE,
            portfolio VARCHAR,
            ticker VARCHAR,
            weight_before DOUBLE,
            weight_after DOUBLE,
            cost DOUBLE,
            PRIMARY KEY (trade_date, portfolio, ticker)
        )
    """)
    # create the asset returns table
    con.execute("""
        CREATE TABLE IF NOT EXISTS asset_returns (
            date DATE,
            ticker VARCHAR,
            daily_return DOUBLE,
            PRIMARY KEY (date, ticker)
        )
    """)

In [10]:
# add the rows of a dataframe to one of the live tables
# will raise an error if a key is in the table already
def append_rows(con, table, rows):
    con.execute(f"INSERT INTO {table} BY NAME SELECT * FROM rows")

In [11]:
# function trial
new_rows = pd.DataFrame({"date": ["2026-12-01", "2026-12-01"],
                         "portfolio": ["ew", "sixty_forty"],
                         "nav": [1.0, 1.0],
                         "daily_return": [0.0, 0.0]})

append_rows(con, "nav", new_rows)
con.execute("SELECT * FROM nav").df()

,date,portfolio,nav,daily_return
0,2026-12-01,ew,1.0,0.0
1,2026-12-01,sixty_forty,1.0,0.0


In [12]:
# now we connect to the database file, creating the folder and the tables if needed
def open_store(path = LIVE_DB):
    path.parent.mkdir(parents = True, exist_ok = True)
    con = duckdb.connect(str(path))
    create_tables(con)
    return con

In [13]:
# checks atest date recorded in the nav table, NA if nothing is recorded yet
def last_date(con):
    return con.execute("SELECT max(date) FROM nav").fetchone()[0] # fetchnote takes that one row out, as a tuple like

In [14]:
# NAV of each portfolio on the latest recorded date
# returns a dataframe with columns portfolio and nav
def last_nav(con):
    return con.execute("""
        SELECT portfolio, nav
        FROM nav
        WHERE date = (SELECT max(date) FROM nav)
    """).df()

In [15]:
# Weights held by each portfolio at the latest recorded close
# returns a dataframe with columns portfolio, ticker and weight
def last_holdings(con):
    return con.execute("""
        SELECT portfolio, ticker, weight
        FROM holdings
        WHERE date = (SELECT max(date) FROM holdings)
    """).df()

In [16]:
# functions trial
con = duckdb.connect() # in memory is fine for trying
create_tables(con)

last_date(con) # None
last_nav(con) # empty dataframe

day1 = pd.DataFrame({"date": ["2026-12-01"] * 2, "portfolio": ["nco", "ew"],
                     "nav": [1.0, 1.0], "daily_return": [0.0, 0.0]})
day2 = pd.DataFrame({"date": ["2026-12-02"] * 2, "portfolio": ["nco", "ew"],
                     "nav": [1.01, 0.99], "daily_return": [0.01, -0.01]})
append_rows(con, "nav", day1)
append_rows(con, "nav", day2)

last_date(con) # 2026-12-02
last_nav(con) # nco 1.01, ew 0.99: only the second day

,portfolio,nav
0,nco,1.01
1,ew,0.99


In [17]:
tmp_path = Path(tempfile.mkdtemp())
con2 = open_store(tmp_path / "test.duckdb")
con2.execute("SHOW TABLES").df() # five tables
list(tmp_path.iterdir()) # the file is there

[PosixPath('/tmp/tmpvgz96qv9/test.duckdb.wal'),
 PosixPath('/tmp/tmpvgz96qv9/test.duckdb')]

In [18]:
con2.close()

Now we move to testing. Tests are performed after all the functions are built because here they can only be checked together

In [19]:
# helper: nav rows for one day, from a dictionary portfolio -> nav
def make_nav_rows(day, navs):
    return pd.DataFrame({"date": day, "portfolio": list(navs.keys()),
                         "nav": list(navs.values()), "daily_return": 0.0})
# helper: an empty database in memory, with the tables created
def make_store():
    con = duckdb.connect()
    create_tables(con)
    return con

In [20]:
# create_tables
def test_create_tables():
    con = make_store()
    # 1. always true: calling it again does not fail
    create_tables(con)
    # 2. known answer: the five tables exist
    tables = set(con.execute("SHOW TABLES").df()["name"])
    assert tables == {"nav", "holdings", "targets", "trades", "asset_returns"}
    # 3. inputs matter: rows already stored survive another call
    append_rows(con, "nav", make_nav_rows("2026-12-01", {"nco": 1.0}))
    create_tables(con)
    assert con.execute("SELECT count(*) FROM nav").fetchone()[0] == 1

In [21]:
test_create_tables()

In [22]:
# append_rows
def test_append_rows():
    con = make_store()
    append_rows(con, "nav", make_nav_rows("2026-12-01", {"nco": 1.0, "ew": 1.0}))
    # 1. always true: the table grows by the number of rows added
    assert con.execute("SELECT count(*) FROM nav").fetchone()[0] == 2
    # 2. known answer: values land in the right columns, even with the columns in another order
    shuffled = make_nav_rows("2026-12-02", {"nco": 1.05})[["nav", "daily_return", "portfolio", "date"]]
    append_rows(con, "nav", shuffled)
    stored = con.execute("SELECT portfolio, nav FROM nav WHERE date = '2026-12-02'").fetchone()
    assert stored == ("nco", 1.05)
    # 3. inputs matter: a row already there is refused, and nothing from that batch is added
    batch = make_nav_rows("2026-12-02", {"nco": 9.99, "ew": 1.0}) # nco repeated, ew new
    with pytest.raises(duckdb.ConstraintException):
        append_rows(con, "nav", batch)
    assert con.execute("SELECT count(*) FROM nav").fetchone()[0] == 3
    assert con.execute("SELECT nav FROM nav WHERE date = '2026-12-02' AND portfolio = 'nco'").fetchone()[0] == 1.05

In [23]:
test_append_rows()

In [24]:
# last_date, last_nav, last_holdings
def test_read_functions():
    con = make_store()
    # 1. always true: an empty store answers without failing
    assert last_date(con) is None
    assert last_nav(con).empty and last_holdings(con).empty
    # 2. known answer: with two days stored, the answers come from the later one
    append_rows(con, "nav", make_nav_rows("2026-12-01", {"nco": 1.0, "ew": 1.0}))
    append_rows(con, "nav", make_nav_rows("2026-12-02", {"nco": 1.01, "ew": 0.99}))
    assert last_date(con) == date(2026, 12, 2)
    navs = last_nav(con).set_index("portfolio")["nav"]
    assert navs["nco"] == 1.01 and navs["ew"] == 0.99 and len(navs) == 2
    for day, weight in [("2026-12-01", 0.5), ("2026-12-02", 0.6)]:
        append_rows(con, "holdings", pd.DataFrame({"date": day, "portfolio": "nco",
                                                   "ticker": ["SPY", "AGG"], "weight": [weight, 1 - weight]}))
    held = last_holdings(con).set_index("ticker")["weight"]
    assert held["SPY"] == 0.6 and len(held) == 2
    # 3. inputs matter: adding a later day moves the answer
    append_rows(con, "nav", make_nav_rows("2026-12-03", {"nco": 1.02, "ew": 0.98}))
    assert last_date(con) == date(2026, 12, 3)

In [25]:
test_read_functions()

In [26]:
# open_store
def test_open_store(tmp_path):
    path = tmp_path / "live" / "test.duckdb" # the folder does not exist yet
    con = open_store(path)
    # 1. always true: the file is created, with its folder
    assert path.exists()
    # 2. known answer: the tables are ready
    assert len(con.execute("SHOW TABLES").df()) == 5
    # 3. inputs matter: what is stored is still there after closing and opening again
    append_rows(con, "nav", make_nav_rows("2026-12-01", {"nco": 1.0}))
    con.close()
    again = open_store(path)
    assert last_date(again) == date(2026, 12, 1)
    again.close()

In [27]:
test_open_store(Path(tempfile.mkdtemp()))